### Breaks vs Switch probability

In [ ]:
from scipy import stats
import mab_subjects
import numpy as np
import pandas as pd

exps = (
    mab_subjects.unstruc.p8020_good_intact_sess
    + mab_subjects.struc.p8020_good_intact_sess
)


df_main = []

for i, exp in enumerate(exps):
    print(exp.sub_name)

    task = exp.b2a
    datetimes = task.datetime
    breaks = np.diff(datetimes) / np.timedelta64(1, "s")  # convert to seconds
    choices = task.choices
    switches = np.diff(choices) != 0  # True where a switch occurs

    bins_breaks = np.arange(0, 100, 1)
    swp = stats.binned_statistic(breaks, switches, statistic="mean", bins=bins_breaks)

    # task_filt = task.filter_by_trials(min_trials=100, clip_max=100)
    # perf_overall = task_filt.get_optimal_choice_probability()
    dict_temp = {
        "bins_breaks": swp.bin_edges[:-1],
        "switch_prob": swp.statistic,
        "sub_name": exp.sub_name,
    }
    dict_temp.update(exp.common_kwargs)
    df_main.append(pd.DataFrame([dict_temp]))

df_main = pd.concat(df_main, ignore_index=True)
mab_subjects.GroupData().save(df_main, "breaks_vs_swp")